# LiDAR Gap Analysis — Puerto Rico G-LiHT
**Luquillo Experimental Forest** | Epochs: March 2017 (pre-Maria) · May 2018 (post-Maria) · March 2020 (recovery)

Pipeline:
1. Load & align LAS files across epochs
2. Compute absolute C2C nearest-neighbour distances
3. Extract connected components to isolate damage clumps
4. Power-law gap size distribution analysis
5. Epoch comparison (damage vs recovery)

In [ ]:
import os
import re
import csv
import sys
import time
import gzip
import shutil
import logging
import tempfile
from math import log2, sqrt
from pathlib import Path
from typing import Any

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats as sp_stats

import cloudComPy as cc
cc.initCC()
print('[INIT] CloudComPy initialized')

## 1. Configuration

In [ ]:
# ── Input directories ────────────────────────────────────────────────────────
DIR_2017 = Path('/ibstorage/shared/PR_NASA_GLiHT/PR_11March2017_FIA12/lidar/las')
DIR_2018 = Path('/ibstorage/shared/PR_NASA_GLiHT/PR_2May018_FIA12/lidar/las')
DIR_2020 = Path('/ibstorage/shared/PR_NASA_GLiHT/PR_15March2020_FIA12/lidar/las')

# ── Output directories ───────────────────────────────────────────────────────
OUT_DIR  = Path('/home/ogh6/Downloads/raw_c2c_output')
CSV_DIR  = OUT_DIR / 'csv'
PLOT_DIR = OUT_DIR / 'plots'
BIN_DIR  = OUT_DIR / 'bin'
for d in [OUT_DIR, CSV_DIR, PLOT_DIR, BIN_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Processing parameters ────────────────────────────────────────────────────
USE_SUBSAMPLE  = True
SUBSAMPLE_N    = 5_000_000
THREADS        = 12

# ── C2C analysis parameters ──────────────────────────────────────────────────
NOISE_CEIL_M   = 20.0
C2C_THRESHOLDS = [2, 5, 10, 15]
EXTREME_PCTILE = 95
HIST_MAX_M     = 20.0
HIST_BINS      = 200

# ── Connected component parameters ───────────────────────────────────────────
CC_CONFIG = {
    'damage_threshold':     2.0,
    'recovery_threshold':   2.0,
    'min_component_pts':    100,
    'min_gap_area_m2':      10.0,
    'overlap_min_fraction': 0.80,
    'overlap_buffer_m':     50.0,
    'max_components':       100_000,
}

GRID_RE  = re.compile(r'(l\d+s\d+)', re.IGNORECASE)
_TMP_DIR = Path(tempfile.gettempdir()) / f'cc_c2c_{int(time.time())}'
_TMP_DIR.mkdir(parents=True, exist_ok=True)

print('Configuration loaded.')
print(f'  2017 : {DIR_2017}\n  2018 : {DIR_2018}\n  2020 : {DIR_2020}\n  Out  : {OUT_DIR}')

## 2. Utility Functions

In [ ]:
def tnow(): return time.time()
def fmt_s(dt):
    if dt < 60:   return f'{dt:.1f}s'
    if dt < 3600: return f'{dt/60:.1f}m'
    return f'{dt/3600:.2f}h'

def mem_gb():
    try:
        import psutil
        return psutil.Process(os.getpid()).memory_info().rss / (1024**3)
    except ImportError:
        return -1.0

def gunzip_if_needed(p):
    p = Path(p)
    if p.suffix != '.gz': return p
    out = _TMP_DIR / p.name[:-3]
    if out.exists() and out.stat().st_size > 0: return out
    print(f'  [UNZIP] {p.name} ...', flush=True)
    t0 = tnow()
    with gzip.open(p, 'rb') as fin, open(out, 'wb') as fout:
        shutil.copyfileobj(fin, fout)
    print(f'  [UNZIP] done {fmt_s(tnow()-t0)}', flush=True)
    return out

def grid_id_from_name(p):
    m = GRID_RE.search(Path(p).name)
    if not m: raise ValueError(f'No grid ID in {p}')
    return m.group(1).lower()

def find_match(folder, grid):
    for f in Path(folder).glob('*.las*'):
        m = GRID_RE.search(f.name)
        if m and m.group(1).lower() == grid: return f
    return None

def load_cloud(p, shift=None):
    t0 = tnow()
    print(f'  [LOAD] {Path(p).name} ...', flush=True)
    if shift is not None:
        cloud = cc.loadPointCloud(filename=str(p), mode=cc.CC_SHIFT_MODE.XYZ,
                                  x=shift[0], y=shift[1], z=shift[2])
    else:
        cloud = cc.loadPointCloud(str(p))
    if cloud is None: raise RuntimeError(f'Failed to load {p}')
    applied_shift = (0.0, 0.0, 0.0)
    if cloud.isShifted():
        gs = cloud.getGlobalShift()
        applied_shift = (float(gs[0]), float(gs[1]), float(gs[2]))
    print(f'  [LOAD] done {fmt_s(tnow()-t0)} | pts={cloud.size():,} | shift={applied_shift} | mem={mem_gb():.2f}GB', flush=True)
    return cloud, applied_shift

def safe_delete(entity):
    try:
        if entity is not None: cc.deleteEntity(entity)
    except Exception:
        pass

print('Utility functions ready.')

## 3. Normalisation, Subsampling & C2C

In [ ]:
def check_normalization(cloud, label=''):
    bb = cloud.getOwnBB()
    z_min, z_max = float(bb.minCorner()[2]), float(bb.maxCorner()[2])
    sf_names = list(cloud.getScalarFieldDic().keys())
    has_class = any('classif' in n.lower() or 'class' in n.lower() for n in sf_names)
    if z_min > 50:     normalised, note = False, f'Z min={z_min:.1f}m — raw GPS. C2C still valid.'
    elif -2 <= z_min < 5: normalised, note = True, 'Appears height-normalised.'
    else:              normalised, note = None, f'Ambiguous (z_min={z_min:.1f}). Check manually.'
    return {'label': label, 'z_min': z_min, 'z_max': z_max,
            'has_classification': has_class, 'likely_normalised': normalised,
            'note': note, 'scalar_fields': ', '.join(sf_names)}

def print_norm_report(info):
    status = {True: 'YES', False: 'NO', None: 'UNCLEAR'}[info['likely_normalised']]
    print(f"  [NORM] {info['label']}: z_min={info['z_min']:.1f} z_max={info['z_max']:.1f} | normalised={status}")
    if info['likely_normalised'] is False: print(f"         {info['note']}")

def subsample_cloud(cloud, target_n):
    if cloud.size() <= target_n: return cloud
    t0 = tnow()
    print(f'  [SUB] {cloud.size():,} -> ~{target_n:,} ...', flush=True)
    ref = cc.CloudSamplingTools.subsampleCloudWithOctree(
        cloud, int(target_n), cc.SUBSAMPLING_CELL_METHOD.NEAREST_POINT_TO_CELL_CENTER)
    sub, res = cloud.partialClone(ref)
    if res != 0: raise RuntimeError(f'partialClone failed (code {res})')
    print(f'  [SUB] done {fmt_s(tnow()-t0)} | out={sub.size():,} | mem={mem_gb():.2f}GB', flush=True)
    return sub

def compute_c2c(src, tgt, sf_name, threads=12):
    t0 = tnow()
    print(f'  [C2C] {sf_name}: approx pass ...', flush=True)
    approx_stats = cc.DistanceComputationTools.computeApproxCloud2CloudDistance(src, tgt)
    if approx_stats:
        print(f'  [C2C] approx: min={approx_stats[0]:.2f} max={approx_stats[1]:.2f} mean={approx_stats[2]:.2f}', flush=True)
    best_level = cc.DistanceComputationTools.determineBestOctreeLevel(src, None, tgt)
    print(f'  [C2C] octree level = {best_level}', flush=True)
    params = cc.Cloud2CloudDistancesComputationParams()
    params.octreeLevel = best_level; params.multiThread = True; params.maxThreadCount = int(threads)
    ret = cc.DistanceComputationTools.computeCloud2CloudDistances(src, tgt, params)
    if ret < 0: print(f'  [C2C] WARNING: returned {ret}', flush=True)
    sf = src.getScalarField(src.getNumberOfScalarFields() - 1)
    sf.setName(sf_name); sf.computeMinAndMax()
    for name, idx in src.getScalarFieldDic().items():
        if 'approx' in name.lower():
            src.deleteScalarField(idx); break
    print(f'  [C2C] {sf_name} done in {fmt_s(tnow()-t0)}', flush=True)
    return sf_name

def get_sf_as_array(cloud, sf_name):
    sf = cloud.getScalarField(sf_name)
    if sf is None:
        d = cloud.getScalarFieldDic()
        if sf_name in d: sf = cloud.getScalarField(d[sf_name])
    if sf is None:
        raise KeyError(f"SF '{sf_name}' not found. Available: {list(cloud.getScalarFieldDic().keys())}")
    return np.asarray(sf.toNpArrayCopy(), dtype=np.float64)

print('Normalisation, subsampling, C2C functions ready.')

## 4. C2C Statistical Analysis & Plotting

In [ ]:
def c2c_summary_stats(c2c, label=''):
    clean = c2c[c2c <= NOISE_CEIL_M]; n_total = len(c2c); n_clean = len(clean)
    out = {'label': label, 'n_total': n_total, 'n_clean': n_clean,
           'n_noise': n_total-n_clean, 'pct_noise': (n_total-n_clean)/max(n_total,1)*100}
    if n_clean > 0:
        out.update({'mean': float(np.mean(clean)), 'median': float(np.median(clean)),
                    'std':  float(np.std(clean)),  'min':    float(np.min(clean)),
                    'max':  float(np.max(clean)),  'p05':    float(np.percentile(clean,5)),
                    'p25':  float(np.percentile(clean,25)), 'p75': float(np.percentile(clean,75)),
                    'p95':  float(np.percentile(clean,95)), 'p99': float(np.percentile(clean,99))})
    return out

def c2c_threshold_fractions(c2c, thresholds=C2C_THRESHOLDS):
    clean = c2c[c2c <= NOISE_CEIL_M]; n = max(len(clean),1)
    return {t: float(np.sum(clean >= t)/n) for t in thresholds}

def extreme_value_report(c2c, label='', percentile=EXTREME_PCTILE):
    clean = c2c[c2c <= NOISE_CEIL_M]
    if len(clean) == 0: return {'label': label, 'error': 'no clean points'}
    thresh = float(np.percentile(clean, percentile)); extreme = clean[clean > thresh]
    return {'label': label, 'percentile': percentile, 'threshold_m': thresh,
            'n_extreme': len(extreme), 'n_clean': len(clean),
            'extreme_mean': float(np.mean(extreme)), 'extreme_median': float(np.median(extreme)),
            'extreme_std':  float(np.std(extreme)),  'extreme_max':    float(np.max(extreme)),
            'pct_2_5m':   float(np.sum((extreme>=2) &(extreme<5)) /max(len(extreme),1)*100),
            'pct_5_10m':  float(np.sum((extreme>=5) &(extreme<10))/max(len(extreme),1)*100),
            'pct_10_15m': float(np.sum((extreme>=10)&(extreme<15))/max(len(extreme),1)*100),
            'pct_15_20m': float(np.sum((extreme>=15)&(extreme<20))/max(len(extreme),1)*100)}

def statistical_comparison(c2c_a, c2c_b, label_a='A', label_b='B'):
    a = c2c_a[c2c_a <= NOISE_CEIL_M]; b = c2c_b[c2c_b <= NOISE_CEIL_M]
    ks_stat, ks_p = sp_stats.ks_2samp(a, b)
    max_n = 500_000
    a_t = np.random.choice(a, min(len(a),max_n), replace=False) if len(a)>max_n else a
    b_t = np.random.choice(b, min(len(b),max_n), replace=False) if len(b)>max_n else b
    mw_stat, mw_p = sp_stats.mannwhitneyu(a_t, b_t, alternative='two-sided')
    result = {'comparison': f'{label_a} vs {label_b}',
              f'{label_a}_n': len(a), f'{label_b}_n': len(b),
              'ks_statistic': float(ks_stat), 'ks_pvalue': float(ks_p),
              'mannwhitney_U': float(mw_stat), 'mannwhitney_pvalue': float(mw_p)}
    for tag, arr in [(label_a,a),(label_b,b)]:
        result[f'{tag}_mean'] = float(np.mean(arr)); result[f'{tag}_median'] = float(np.median(arr))
        result[f'{tag}_std']  = float(np.std(arr))
    return result

def plot_grid(grid_id, c2c_1718, c2c_1820, out_dir=PLOT_DIR):
    clean_1718 = c2c_1718[c2c_1718<=NOISE_CEIL_M]; clean_1820 = c2c_1820[c2c_1820<=NOISE_CEIL_M]
    fig, axes = plt.subplots(2,3,figsize=(18,10))
    fig.suptitle(f'Grid {grid_id.upper()} — Raw C2C Displacement', fontsize=14, fontweight='bold')
    edges = np.linspace(0,HIST_MAX_M,HIST_BINS+1)
    ax=axes[0,0]; ax.hist(clean_1718,bins=edges,density=True,alpha=0.5,color='red', label='2017→2018')
    ax.hist(clean_1820,bins=edges,density=True,alpha=0.5,color='blue',label='2018→2020')
    ax.set_xlabel('C2C (m)');ax.set_ylabel('Density');ax.set_title('(a) Distribution');ax.legend(fontsize=8);ax.set_xlim(0,HIST_MAX_M)
    ax=axes[0,1]
    for arr,lbl,clr in [(clean_1718,'2017→2018','red'),(clean_1820,'2018→2020','blue')]:
        s=np.sort(arr);cdf=np.arange(1,len(s)+1)/len(s);step=max(len(s)//10000,1)
        ax.plot(s[::step],cdf[::step],color=clr,lw=1.5,label=lbl)
    ax.set_xlabel('C2C (m)');ax.set_ylabel('CDF');ax.set_title('(b) CDF');ax.legend(fontsize=8);ax.set_xlim(0,HIST_MAX_M);ax.grid(True,alpha=0.3)
    ax=axes[0,2]; x_pos=np.arange(len(C2C_THRESHOLDS)); width=0.35
    f1=c2c_threshold_fractions(c2c_1718); f2=c2c_threshold_fractions(c2c_1820)
    ax.bar(x_pos-width/2,[f1[t]*100 for t in C2C_THRESHOLDS],width,color='red', alpha=0.7,label='2017→2018')
    ax.bar(x_pos+width/2,[f2[t]*100 for t in C2C_THRESHOLDS],width,color='blue',alpha=0.7,label='2018→2020')
    ax.set_xticks(x_pos);ax.set_xticklabels([f'>={t}m' for t in C2C_THRESHOLDS])
    ax.set_ylabel('% clean pts');ax.set_title('(c) Threshold Exceedance');ax.legend(fontsize=8)
    ax=axes[1,0]; bands=['2-5m','5-10m','10-15m','15-20m']
    ev1=extreme_value_report(c2c_1718);ev2=extreme_value_report(c2c_1820)
    keys=['pct_2_5m','pct_5_10m','pct_10_15m','pct_15_20m']
    ax.bar(np.arange(4)-width/2,[ev1.get(k,0) for k in keys],width,color='red', alpha=0.7,label='2017→2018')
    ax.bar(np.arange(4)+width/2,[ev2.get(k,0) for k in keys],width,color='blue',alpha=0.7,label='2018→2020')
    ax.set_xticks(np.arange(4));ax.set_xticklabels(bands)
    ax.set_ylabel(f'% extreme (>{EXTREME_PCTILE}th pctile)');ax.set_title('(d) Extreme Bands');ax.legend(fontsize=8)
    ax=axes[1,1]; fine=np.linspace(0,5,100)
    ax.hist(clean_1718,bins=fine,density=True,alpha=0.5,color='red', label='2017→2018')
    ax.hist(clean_1820,bins=fine,density=True,alpha=0.5,color='blue',label='2018→2020')
    ax.set_xlabel('C2C (m)');ax.set_ylabel('Density');ax.set_title('(e) Detail 0–5m');ax.legend(fontsize=8)
    ax.axvline(2.0,color='gray',linestyle='--',alpha=0.5)
    ax=axes[1,2]; ax.axis('off')
    s1=c2c_summary_stats(c2c_1718); s2=c2c_summary_stats(c2c_1820)
    txt=(f"2017→2018\n  n={s1['n_clean']:,}  noise={s1['n_noise']:,} ({s1['pct_noise']:.1f}%)\n"
         f"  mean={s1.get('mean',0):.2f}  med={s1.get('median',0):.2f}  std={s1.get('std',0):.2f}\n"
         f"  p95={s1.get('p95',0):.2f}  p99={s1.get('p99',0):.2f}\n\n"
         f"2018→2020\n  n={s2['n_clean']:,}  noise={s2['n_noise']:,} ({s2['pct_noise']:.1f}%)\n"
         f"  mean={s2.get('mean',0):.2f}  med={s2.get('median',0):.2f}  std={s2.get('std',0):.2f}\n"
         f"  p95={s2.get('p95',0):.2f}  p99={s2.get('p99',0):.2f}\n\nNOISE CEIL: {NOISE_CEIL_M}m")
    ax.text(0.05,0.95,txt,transform=ax.transAxes,fontsize=9,va='top',fontfamily='monospace',
            bbox=dict(boxstyle='round',facecolor='wheat',alpha=0.5))
    ax.set_title('(f) Summary')
    plt.tight_layout(rect=[0,0,1,0.95])
    out=Path(out_dir)/f'{grid_id}_c2c_analysis.png'; fig.savefig(str(out),dpi=150); plt.close(fig)
    print(f'  [PLOT] {out.name}', flush=True)

def write_csv(filepath, rows, fieldnames=None):
    if not rows: return
    if fieldnames is None:
        seen = {}
        for r in rows:
            for k in r: seen[k] = True
        fieldnames = list(seen)
    with open(filepath,'w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fieldnames,extrasaction='ignore')
        w.writeheader(); w.writerows(rows)
    print(f'  [CSV] {filepath}', flush=True)

print('C2C analysis and plotting functions ready.')

## 5. Connected Component Functions
Spatial overlap verification, component extraction, power-law fitting, and gap size distribution analysis.

In [ ]:
# ── Config helper ─────────────────────────────────────────────────────────────
def _cfg(config, key):
    if config and key in config: return config[key]
    return CC_CONFIG[key]

# ── Overlap verification ──────────────────────────────────────────────────────
def _bb_xy(cloud):
    bb=cloud.getOwnBB(); lo=bb.minCorner(); hi=bb.maxCorner()
    return (lo[0],lo[1],hi[0],hi[1])

def _intersect_xy(boxes):
    xmin=max(b[0] for b in boxes); ymin=max(b[1] for b in boxes)
    xmax=min(b[2] for b in boxes); ymax=min(b[3] for b in boxes)
    return None if xmin>=xmax or ymin>=ymax else (xmin,ymin,xmax,ymax)

def _box_area(box): return (box[2]-box[0])*(box[3]-box[1])

def verify_and_clip_overlap(clouds_dict, label, buffer_m=50.0, config=None):
    min_frac = _cfg(config,'overlap_min_fraction')
    names = sorted(clouds_dict)
    boxes = {n: _bb_xy(clouds_dict[n]) for n in names}
    areas = {n: _box_area(boxes[n])    for n in names}
    inter = _intersect_xy(list(boxes.values()))
    if inter is None:
        msg=f'[OVERLAP] {label}: NO XY overlap — skipping grid.'
        print(msg); raise ValueError(msg)
    inter_area   = _box_area(inter)
    overlap_frac = inter_area / min(areas.values())
    report = {'intersection_area_m2': inter_area, 'cloud_areas': dict(areas),
              'overlap_fraction': overlap_frac, 'was_clipped': False, 'buffer_used': 0.0}
    if overlap_frac >= min_frac:
        print(f'[OVERLAP] {label}: {overlap_frac:.1%} overlap — no clip needed.')
        return dict(clouds_dict), report
    print(f'[OVERLAP] {label}: {overlap_frac:.1%} overlap — clipping to intersection + {buffer_m}m buffer.')
    z_min = min(clouds_dict[n].getOwnBB().minCorner()[2] for n in names)
    z_max = max(clouds_dict[n].getOwnBB().maxCorner()[2] for n in names)
    clip_box = cc.ccBBox((inter[0]-buffer_m, inter[1]-buffer_m, z_min-1.0),
                         (inter[2]+buffer_m, inter[3]+buffer_m, z_max+1.0), True)
    clipped = {}
    for n in names:
        cropped = cc.cropPointCloud(clouds_dict[n], clip_box, inside=True)
        clipped[n] = clouds_dict[n] if (cropped is None or cropped.size()==0) else cropped
    report['was_clipped']=True; report['buffer_used']=buffer_m
    return clipped, report

# ── Component extraction ──────────────────────────────────────────────────────
def _dynamic_octree_level(cloud):
    bb=cloud.getOwnBB(); lo=bb.minCorner(); hi=bb.maxCorner()
    diag=sqrt((hi[0]-lo[0])**2+(hi[1]-lo[1])**2+(hi[2]-lo[2])**2)
    return max(6, min(12, int(log2(max(diag/2.0,1.0))))) if diag>0 else 8

def extract_damage_components(source_cloud, c2c_array, sf_name, grid_id, epoch_label, config=None):
    threshold = _cfg(config,'damage_threshold')
    min_pts   = _cfg(config,'min_component_pts')
    max_comps = _cfg(config,'max_components')
    n_above   = int(np.sum(c2c_array > threshold))
    if n_above == 0:
        print(f'[COMP] {grid_id}/{epoch_label}: no points exceed {threshold}m — skipping.')
        return [], [], 0
    print(f'[COMP] {grid_id}/{epoch_label}: {n_above:,}/{c2c_array.size:,} pts ({100.0*n_above/c2c_array.size:.1f}%) above {threshold}m.')
    sf_dic = source_cloud.getScalarFieldDic()
    if sf_name not in sf_dic:
        print(f'[COMP] {grid_id}/{epoch_label}: WARNING — SF {sf_name!r} not found. Available: {list(sf_dic)}')
        return [], [], 0
    source_cloud.setCurrentDisplayedScalarField(sf_dic[sf_name])
    max_c2c = float(np.max(c2c_array)) + 1.0
    damaged_cloud = source_cloud.filterPointsByScalarValue(threshold, max_c2c)
    if damaged_cloud is None or damaged_cloud.size()==0:
        print(f'[COMP] {grid_id}/{epoch_label}: WARNING — filter returned empty cloud.')
        return [], [], 0
    print(f'[COMP] {grid_id}/{epoch_label}: damaged_cloud has {damaged_cloud.size():,} pts.')
    octree_level = _dynamic_octree_level(damaged_cloud)
    print(f'[COMP] {grid_id}/{epoch_label}: octree level = {octree_level}.')
    n_proc, components, residuals = cc.ExtractConnectedComponents(
        clouds=[damaged_cloud], octreeLevel=octree_level,
        minComponentSize=min_pts, maxNumberComponents=max_comps, randomColors=False)
    print(f'[COMP] {grid_id}/{epoch_label}: {len(components)} components, {len(residuals)} residual clouds.')
    cc.deleteEntity(damaged_cloud)
    return components, residuals, octree_level

print('Connected component extraction functions ready.')

In [ ]:
# ── Per-component statistics ──────────────────────────────────────────────────
def _sf_array_from_cloud(cloud):
    sf_dict = cloud.getScalarFieldDic()
    if not sf_dict: return None
    sf = cloud.getScalarField(next(iter(sf_dict)))
    return sf.toNpArrayCopy() if sf else None

def compute_component_stats(components, residuals, grid_id, epoch_label, point_density_per_m2=None):
    stats_list = []
    for i, comp in enumerate(components):
        n_pts = int(comp.size())
        bb=comp.getOwnBB(); lo=bb.minCorner(); hi=bb.maxCorner()
        bbox_area = (hi[0]-lo[0])*(hi[1]-lo[1])
        sf_arr = _sf_array_from_cloud(comp)
        if sf_arr is not None and sf_arr.size>0:
            mean_c2c=float(np.mean(sf_arr)); median_c2c=float(np.median(sf_arr))
            std_c2c=float(np.std(sf_arr));   max_c2c=float(np.max(sf_arr))
        else:
            mean_c2c=median_c2c=std_c2c=max_c2c=float('nan')
        rec = {'grid_id':grid_id,'epoch_label':epoch_label,'component_id':i,'n_points':n_pts,
               'bbox_area_m2':bbox_area,'bbox_area_sqrt_m':sqrt(max(bbox_area,0.0)),
               'mean_c2c_m':mean_c2c,'median_c2c_m':median_c2c,'std_c2c_m':std_c2c,'max_c2c_m':max_c2c,
               'centroid_x':0.5*(lo[0]+hi[0]),'centroid_y':0.5*(lo[1]+hi[1])}
        if point_density_per_m2 and point_density_per_m2>0:
            rec['estimated_area_m2'] = n_pts/point_density_per_m2
        stats_list.append(rec)
    total_res_pts = sum(int(r.size()) for r in residuals)
    residual_summary = {'grid_id':grid_id,'epoch_label':epoch_label,
                        'n_residual_clouds':len(residuals),'total_residual_points':total_res_pts}
    print(f'[COMP] {grid_id}/{epoch_label}: stats for {len(stats_list)} components, '
          f'{len(residuals)} residual clouds ({total_res_pts} residual pts).')
    return stats_list, residual_summary

# ── Power law fitting ─────────────────────────────────────────────────────────
def fit_power_law(areas, label='', min_area_m2=10.0):
    areas   = np.asarray(areas, dtype=np.float64)
    fitted  = areas[areas >= min_area_m2]
    result  = {'alpha_mle':float('nan'),'alpha_ols':float('nan'),'xmin':min_area_m2,
               'n_fitted':0,'ks_stat':float('nan'),'ks_pvalue':float('nan'),
               'r2_loglog':float('nan'),'ll_powerlaw':float('nan'),'ll_lognormal':float('nan'),
               'preferred_distribution':'undetermined','areas_fitted':np.array([])}
    if fitted.size < 5:
        print(f'[POWERLAW] {label}: only {fitted.size} areas >= {min_area_m2}m^2 — too few.')
        return result
    n=fitted.size; xmin=min_area_m2; result['n_fitted']=n; result['areas_fitted']=fitted
    # MLE (Clauset et al. 2009)
    sum_log = float(np.sum(np.log(fitted/xmin)))
    alpha_mle = (1.0 + n/sum_log) if sum_log>0 else float('nan')
    result['alpha_mle'] = alpha_mle
    # Log-log OLS
    n_bins = max(10, int(sqrt(n)))
    log_edges = np.linspace(np.log10(xmin), np.log10(fitted.max()), n_bins+1)
    counts, _ = np.histogram(np.log10(fitted), bins=log_edges)
    pos = counts>0
    if pos.sum()>=2:
        slope,_,r,_,_ = sp_stats.linregress(0.5*(log_edges[:-1]+log_edges[1:])[pos],
                                              np.log10(counts[pos].astype(float)))
        result['alpha_ols']=-slope; result['r2_loglog']=r**2
    # KS test
    if np.isfinite(alpha_mle) and alpha_mle>1.0:
        ks = sp_stats.kstest(fitted,'pareto',args=(alpha_mle-1.0,),N=n)
        result['ks_stat']=float(ks.statistic); result['ks_pvalue']=float(ks.pvalue)
        ll_pl = float(n*np.log(alpha_mle-1.0)-n*np.log(xmin)-alpha_mle*np.sum(np.log(fitted/xmin)))
        result['ll_powerlaw']=ll_pl
    else: ll_pl=-np.inf
    # Log-normal comparison
    mu_ln=float(np.mean(np.log(fitted))); sig_ln=float(np.std(np.log(fitted),ddof=1)) if n>1 else 1.0
    ll_ln = float(np.sum(sp_stats.lognorm.logpdf(fitted,s=sig_ln,scale=np.exp(mu_ln)))) if sig_ln>0 else -np.inf
    result['ll_lognormal']=ll_ln
    if np.isfinite(ll_pl) and np.isfinite(ll_ln):
        result['preferred_distribution']='power_law' if ll_pl>ll_ln else 'lognormal'
    tag=f' ({label})' if label else ''
    print(f"[POWERLAW]{tag}: alpha_MLE={alpha_mle:.3f}, alpha_OLS={result['alpha_ols']:.3f}, "
          f"n={n}, KS={result['ks_stat']:.4f}, preferred={result['preferred_distribution']}")
    return result

def compare_power_laws(fit_a, fit_b, label_a, label_b):
    a=fit_a.get('alpha_mle',float('nan')); b=fit_b.get('alpha_mle',float('nan'))
    delta=a-b
    if np.isfinite(delta):
        if   delta<0: interp=f'{label_a} (alpha={a:.2f}) has flatter tail than {label_b} (alpha={b:.2f}) — more large gaps in {label_a}.'
        elif delta>0: interp=f'{label_b} (alpha={b:.2f}) has flatter tail than {label_a} (alpha={a:.2f}) — more large gaps in {label_b}.'
        else:         interp='Identical power-law exponents.'
    else: interp='Cannot compare — at least one fit invalid.'
    print(f'[POWERLAW] {label_a} vs {label_b}: delta_alpha={delta:.3f}')
    return {'label_a':label_a,'label_b':label_b,'alpha_a':a,'alpha_b':b,'delta_alpha':delta,'interpretation':interp}

print('Component stats and power law functions ready.')

In [ ]:
# ── Gap size distribution figures ─────────────────────────────────────────────
def _annotate_fit(ax, fit):
    txt=(f"$\\alpha_{{MLE}}$={fit.get('alpha_mle',float('nan')):.2f}\n"
         f"$R^2_{{OLS}}$={fit.get('r2_loglog',float('nan')):.3f}\n"
         f"n={fit.get('n_fitted',0)}\n"
         f"$x_{{min}}$={fit.get('xmin',float('nan')):.0f} m$^2$")
    ax.text(0.97,0.97,txt,transform=ax.transAxes,fontsize=8,va='top',ha='right',
            bbox=dict(boxstyle='round,pad=0.3',fc='white',alpha=0.8))

def gap_size_distribution_analysis(component_stats, grid_id, epoch_label, out_dir):
    out_dir=Path(out_dir); out_dir.mkdir(parents=True,exist_ok=True)
    areas    = np.array([s['bbox_area_m2'] for s in component_stats],dtype=np.float64)
    mean_c2c = np.array([s['mean_c2c_m']  for s in component_stats],dtype=np.float64)
    fit = fit_power_law(areas, label=f'{grid_id}/{epoch_label}')
    if areas.size==0:
        print(f'[DIST] {grid_id}/{epoch_label}: no components — skipping figure.')
        return fit
    fig,axes=plt.subplots(2,2,figsize=(12,10))
    fig.suptitle(f'Gap Size Distribution — {grid_id} / {epoch_label}',fontsize=13)
    pos_areas=areas[areas>0]
    # (a) Log-log histogram
    ax=axes[0,0]
    if pos_areas.size>0:
        log_bins=np.logspace(np.log10(pos_areas.min()),np.log10(pos_areas.max()),max(10,int(sqrt(pos_areas.size))))
        ax.hist(pos_areas,bins=log_bins,edgecolor='k',alpha=0.7,label='data')
        alpha=fit['alpha_mle']; xmin=fit['xmin']
        if np.isfinite(alpha) and alpha>1.0:
            x_line=np.logspace(np.log10(xmin),np.log10(pos_areas.max()),200)
            pdf=((alpha-1.0)/xmin)*(x_line/xmin)**(-alpha)
            ax.plot(x_line,pdf*np.sum(pos_areas>=xmin)*float(np.mean(np.diff(log_bins))),'r-',lw=2,label=f'PL (a={alpha:.2f})')
    ax.set_xscale('log');ax.set_yscale('log');ax.set_xlabel('Gap area (m²)');ax.set_ylabel('Count')
    ax.set_title('(a) Log-log histogram');ax.legend(fontsize=8);_annotate_fit(ax,fit)
    # (b) Empirical vs theoretical CDF
    ax=axes[0,1]; s=np.sort(areas); ecdf=np.arange(1,len(s)+1)/len(s)
    ax.step(s,ecdf,where='post',label='Empirical CDF',lw=1.5)
    alpha=fit['alpha_mle']; xmin=fit['xmin']
    if np.isfinite(alpha) and alpha>1.0 and xmin>0:
        x_th=np.sort(s[s>=xmin])
        if x_th.size>0:
            tcdf=1.0-(x_th/xmin)**(-(alpha-1.0))
            frac_below=np.searchsorted(s,xmin)/len(s)
            ax.plot(x_th,frac_below+(1.0-frac_below)*tcdf,'r--',lw=1.5,label='Power-law CDF')
    ax.set_xscale('log');ax.set_xlabel('Gap area (m²)');ax.set_ylabel('CDF')
    ax.set_title('(b) CDF comparison');ax.legend(fontsize=8)
    # (c) Area vs mean C2C scatter
    ax=axes[1,0]; valid=np.isfinite(mean_c2c)&(areas>0)
    if valid.any(): ax.scatter(areas[valid],mean_c2c[valid],s=8,alpha=0.5)
    ax.set_xscale('log');ax.set_xlabel('Gap area (m²)');ax.set_ylabel('Mean C2C (m)')
    ax.set_title('(c) Gap area vs C2C displacement')
    # (d) Zipf rank-size plot
    ax=axes[1,1]; ranks=np.arange(1,len(s)+1)
    ax.scatter(ranks,s[::-1],s=8,alpha=0.6)
    ax.set_xscale('log');ax.set_yscale('log');ax.set_xlabel('Rank');ax.set_ylabel('Gap area (m²)')
    ax.set_title('(d) Rank-size (Zipf) plot')
    fig.tight_layout(rect=[0,0,1,0.95])
    fname=out_dir/f'{grid_id}_{epoch_label}_gap_distribution.png'
    fig.savefig(fname,dpi=150); plt.close(fig)
    print(f'[DIST] {grid_id}/{epoch_label}: saved → {fname}')
    return fit

def compare_epochs(stats_1718, stats_1820, grid_id, out_dir):
    out_dir=Path(out_dir); out_dir.mkdir(parents=True,exist_ok=True)
    a17=np.array([s['bbox_area_m2'] for s in stats_1718],dtype=np.float64)
    a18=np.array([s['bbox_area_m2'] for s in stats_1820],dtype=np.float64)
    fit17=fit_power_law(a17,label=f'{grid_id}/damage_1718')
    fit18=fit_power_law(a18,label=f'{grid_id}/recovery_1820')
    pl_cmp=compare_power_laws(fit17,fit18,'damage_1718','recovery_1820')
    ks_r=mw_r={'statistic':float('nan'),'pvalue':float('nan')}
    if a17.size>0 and a18.size>0:
        ks=sp_stats.ks_2samp(a17,a18); ks_r={'statistic':float(ks.statistic),'pvalue':float(ks.pvalue)}
        mw=sp_stats.mannwhitneyu(a17,a18,alternative='two-sided'); mw_r={'statistic':float(mw.statistic),'pvalue':float(mw.pvalue)}
    def sc(arr): return {'lt_100':int(np.sum(arr<100)),'100_to_500':int(np.sum((arr>=100)&(arr<500))),'gt_500':int(np.sum(arr>=500))}
    comparison={'grid_id':grid_id,
                'n_gaps_damage':int(a17.size),'n_gaps_recovery':int(a18.size),
                'mean_gap_area_damage_m2':float(np.mean(a17)) if a17.size else 0,
                'mean_gap_area_recovery_m2':float(np.mean(a18)) if a18.size else 0,
                'total_damaged_area_damage_m2':float(np.sum(a17)),'total_damaged_area_recovery_m2':float(np.sum(a18)),
                'size_classes_damage':sc(a17),'size_classes_recovery':sc(a18),
                'ks_2samp':ks_r,'mann_whitney_u':mw_r,
                'power_law_damage':{k:v for k,v in fit17.items() if k!='areas_fitted'},
                'power_law_recovery':{k:v for k,v in fit18.items() if k!='areas_fitted'},
                'power_law_comparison':pl_cmp}
    # Comparison figure
    fig,axes=plt.subplots(1,3,figsize=(16,5))
    fig.suptitle(f'Epoch Comparison — {grid_id}',fontsize=13)
    all_a=np.concatenate([a17[a17>0],a18[a18>0]])
    if all_a.size>0:
        log_bins=np.logspace(np.log10(max(all_a.min(),0.1)),np.log10(all_a.max()),30)
        ax=axes[0]
        if a17[a17>0].size>0: ax.hist(a17[a17>0],bins=log_bins,alpha=0.5,edgecolor='k',label='Damage 17-18')
        if a18[a18>0].size>0: ax.hist(a18[a18>0],bins=log_bins,alpha=0.5,edgecolor='k',label='Recovery 18-20')
        for fit,color,lbl in [(fit17,'red','PL damage'),(fit18,'blue','PL recovery')]:
            alpha=fit['alpha_mle']; xmin=fit['xmin']
            if np.isfinite(alpha) and alpha>1.0 and all_a.max()>xmin:
                x_l=np.logspace(np.log10(xmin),np.log10(all_a.max()),200)
                ax.plot(x_l,((alpha-1.0)/xmin)*(x_l/xmin)**(-alpha)*fit['n_fitted']*float(np.mean(np.diff(log_bins))),
                        color=color,lw=2,label=f'{lbl} (a={alpha:.2f})')
        ax.set_xscale('log');ax.set_yscale('log');ax.set_xlabel('Gap area (m²)');ax.set_ylabel('Count')
        ax.set_title('(a) Overlaid histograms');ax.legend(fontsize=7)
    ax=axes[1]
    for arr,lbl,clr in [(a17,'Damage 17-18','red'),(a18,'Recovery 18-20','blue')]:
        if arr.size>0:
            sv=np.sort(arr); ax.step(sv,np.arange(1,len(sv)+1)/len(sv),where='post',label=lbl,color=clr,lw=1.5)
    ax.set_xscale('log');ax.set_xlabel('Gap area (m²)');ax.set_ylabel('CDF')
    ax.set_title('(b) Empirical CDFs');ax.legend(fontsize=8)
    ax=axes[2]
    classes=['< 100','100–500','> 500']
    c17=[int(np.sum(a17<100)),int(np.sum((a17>=100)&(a17<500))),int(np.sum(a17>=500))]
    c18=[int(np.sum(a18<100)),int(np.sum((a18>=100)&(a18<500))),int(np.sum(a18>=500))]
    x=np.arange(3); w=0.35
    ax.bar(x-w/2,c17,w,label='Damage 17-18',alpha=0.8); ax.bar(x+w/2,c18,w,label='Recovery 18-20',alpha=0.8)
    ax.set_xticks(x);ax.set_xticklabels(classes);ax.set_ylabel('Count')
    ax.set_title('(c) Gap count by size class');ax.legend(fontsize=8)
    fig.tight_layout(rect=[0,0,1,0.94])
    fname=out_dir/f'{grid_id}_epoch_comparison.png'; fig.savefig(fname,dpi=150); plt.close(fig)
    print(f'[DIST] {grid_id}: epoch comparison → {fname}')
    return comparison

# ── Main CC entry point ───────────────────────────────────────────────────────
def _estimate_point_density(cloud):
    bb=cloud.getOwnBB(); lo=bb.minCorner(); hi=bb.maxCorner()
    xy=(hi[0]-lo[0])*(hi[1]-lo[1])
    return cloud.size()/xy if xy>0 else 0.0

def _delete_cloud_list(clouds, label):
    for c in clouds:
        try: cc.deleteEntity(c)
        except Exception as e: print(f'[COMP] WARNING — delete failed ({label}): {e}')
    clouds.clear()

def run_connected_component_analysis(c2017, c2018, c2020, c2c_1718, c2c_1820, grid_id, config, out_dirs):
    plots_dir=Path(out_dirs.get('plots','outputs/plots'))
    try:
        clipped,overlap_report=verify_and_clip_overlap({'2017':c2017,'2018':c2018,'2020':c2020},
                                                        grid_id,buffer_m=_cfg(config,'overlap_buffer_m'),config=config)
    except ValueError:
        return {'grid_id':grid_id,'status':'skipped_no_overlap','overlap_report':None}
    c2017=clipped['2017']; c2018=clipped['2018']; c2020=clipped['2020']
    density=_estimate_point_density(c2017)
    print(f'[COMP] {grid_id}: point density = {density:.1f} pts/m²')
    results={'grid_id':grid_id,'status':'ok','overlap_report':overlap_report,'point_density_per_m2':density}
    # Damage epoch
    comps17,resid17,oct17=extract_damage_components(c2017,c2c_1718,'C2C_2017_2018',grid_id,'damage_1718',config)
    stats_1718,rsumm17=compute_component_stats(comps17,resid17,grid_id,'damage_1718',density)
    fit17=gap_size_distribution_analysis(stats_1718,grid_id,'damage_1718',plots_dir)
    _delete_cloud_list(comps17,'damage_1718 components'); _delete_cloud_list(resid17,'damage_1718 residuals')
    results['damage_1718']={'component_stats':stats_1718,'residual_summary':rsumm17,
                            'power_law_fit':{k:v for k,v in fit17.items() if k!='areas_fitted'},'octree_level':oct17}
    # Recovery epoch
    comps18,resid18,oct18=extract_damage_components(c2018,c2c_1820,'C2C_2018_2020',grid_id,'recovery_1820',config)
    stats_1820,rsumm18=compute_component_stats(comps18,resid18,grid_id,'recovery_1820',density)
    fit18=gap_size_distribution_analysis(stats_1820,grid_id,'recovery_1820',plots_dir)
    _delete_cloud_list(comps18,'recovery_1820 components'); _delete_cloud_list(resid18,'recovery_1820 residuals')
    results['recovery_1820']={'component_stats':stats_1820,'residual_summary':rsumm18,
                              'power_law_fit':{k:v for k,v in fit18.items() if k!='areas_fitted'},'octree_level':oct18}
    results['epoch_comparison']=compare_epochs(stats_1718,stats_1820,grid_id,plots_dir)
    print(f'[COMP] {grid_id}: connected-component analysis complete.')
    return results

print('All connected component functions ready.')

## 6. Determine Global Coordinate Shift

In [ ]:
files_2017 = sorted(DIR_2017.glob('*.las*'))
print(f'Found {len(files_2017)} files in 2017 directory')

global_shift = None
for probe_file in files_2017:
    try: grid_id_from_name(probe_file)
    except ValueError: continue
    p_probe = gunzip_if_needed(probe_file)
    print(f'[SHIFT] Probing {p_probe.name} ...')
    probe_cloud = cc.loadPointCloud(str(p_probe))
    if probe_cloud is None: continue
    global_shift = tuple(float(x) for x in probe_cloud.getGlobalShift()) if probe_cloud.isShifted() else (0.0,0.0,0.0)
    bb=probe_cloud.getOwnBB()
    print(f'[SHIFT] shift={global_shift}')
    print(f'[SHIFT] BB: X=[{float(bb.minCorner()[0]):.1f},{float(bb.maxCorner()[0]):.1f}] '
          f'Y=[{float(bb.minCorner()[1]):.1f},{float(bb.maxCorner()[1]):.1f}] '
          f'Z=[{float(bb.minCorner()[2]):.1f},{float(bb.maxCorner()[2]):.1f}]')
    safe_delete(probe_cloud)
    break

if global_shift is None: raise RuntimeError('Could not determine shift from any 2017 file')
print(f'\nUsing global shift {global_shift} for ALL files.')

## 7. Main Processing Loop

In [ ]:
all_summaries=[];all_thresholds=[];all_tests=[];all_extreme=[];all_norm=[];all_cc_results=[]
n_processed=n_skip_2018=n_skip_2020=0; norm_done=False

for idx, f2017 in enumerate(files_2017, start=1):
    loop_t0=tnow()
    try: grid=grid_id_from_name(f2017)
    except ValueError: continue
    f2018=find_match(DIR_2018,grid)
    if f2018 is None: n_skip_2018+=1; continue
    f2020=find_match(DIR_2020,grid)
    if f2020 is None: n_skip_2020+=1; continue

    print(f'\n{"="*70}\n  [{idx}/{len(files_2017)}]  GRID {grid.upper()}\n{"="*70}')

    p2017=gunzip_if_needed(f2017); p2018=gunzip_if_needed(f2018); p2020=gunzip_if_needed(f2020)
    c2017,_=load_cloud(p2017,shift=global_shift)
    c2018,_=load_cloud(p2018,shift=global_shift)
    c2020,_=load_cloud(p2020,shift=global_shift)

    for lbl,c in [('2018',c2018),('2020',c2020)]:
        br=c2017.getOwnBB(); bo=c.getOwnBB()
        dist=abs(float(br.minCorner()[0])-float(bo.minCorner()[0]))+abs(float(br.minCorner()[1])-float(bo.minCorner()[1]))
        print(f'  [CHECK] 2017 vs {lbl} BB offset: {dist:.1f}m ({"OK" if dist<=100 else "WARN"})')

    if not norm_done:
        for c,lbl in [(c2017,f'{grid}_2017'),(c2018,f'{grid}_2018'),(c2020,f'{grid}_2020')]:
            info=check_normalization(c,lbl); print_norm_report(info); all_norm.append(info)
        norm_done=True
    else:
        info=check_normalization(c2017,f'{grid}_2017'); all_norm.append(info)

    did_sub=False
    if USE_SUBSAMPLE:
        c2017s=subsample_cloud(c2017,SUBSAMPLE_N); c2018s=subsample_cloud(c2018,SUBSAMPLE_N)
        c2020s=subsample_cloud(c2020,SUBSAMPLE_N); did_sub=(c2017s is not c2017)
    else:
        c2017s=c2017; c2018s=c2018; c2020s=c2020

    sf_1718=compute_c2c(c2017s,c2018s,'C2C_2017_2018',threads=THREADS)
    c2c_1718=get_sf_as_array(c2017s,sf_1718)
    cc.SavePointCloud(c2017s,str(BIN_DIR/f'{grid}_2017_c2c.bin'))
    print(f'  [DATA] 1718: {len(c2c_1718):,} pts')

    if did_sub and c2017s is not c2017: safe_delete(c2017s); c2017s=None
    safe_delete(c2017); c2017=None
    print(f'  [MEM] freed 2017 | mem={mem_gb():.2f}GB')

    sf_1820=compute_c2c(c2018s,c2020s,'C2C_2018_2020',threads=THREADS)
    c2c_1820=get_sf_as_array(c2018s,sf_1820)
    cc.SavePointCloud(c2018s,str(BIN_DIR/f'{grid}_2018_c2c.bin'))
    print(f'  [DATA] 1820: {len(c2c_1820):,} pts')

    s1=c2c_summary_stats(c2c_1718,f'{grid}_2017to2018'); s1['grid']=grid; s1['interval']='2017-2018'
    s2=c2c_summary_stats(c2c_1820,f'{grid}_2018to2020'); s2['grid']=grid; s2['interval']='2018-2020'
    all_summaries.extend([s1,s2])
    tf1=c2c_threshold_fractions(c2c_1718); tf2=c2c_threshold_fractions(c2c_1820)
    row={'grid':grid}
    for t in C2C_THRESHOLDS: row[f'1718_gte_{t}m']=tf1[t]; row[f'1820_gte_{t}m']=tf2[t]
    all_thresholds.append(row)
    test=statistical_comparison(c2c_1718,c2c_1820,'2017_2018','2018_2020'); test['grid']=grid; all_tests.append(test)
    ev1=extreme_value_report(c2c_1718,f'{grid}_2017to2018'); ev1['grid']=grid; ev1['interval']='2017-2018'
    ev2=extreme_value_report(c2c_1820,f'{grid}_2018to2020'); ev2['grid']=grid; ev2['interval']='2018-2020'
    all_extreme.extend([ev1,ev2])
    plot_grid(grid,c2c_1718,c2c_1820)

    # Connected components (reload 2017 from saved .bin)
    c2017s_bin=cc.loadPointCloud(str(BIN_DIR/f'{grid}_2017_c2c.bin'))
    cc_results=run_connected_component_analysis(
        c2017s_bin,c2018s,c2020s,c2c_1718,c2c_1820,
        grid_id=grid,config=CC_CONFIG,out_dirs={'plots':PLOT_DIR})
    all_cc_results.append(cc_results)
    safe_delete(c2017s_bin)

    if did_sub: safe_delete(c2018s); safe_delete(c2020s)
    safe_delete(c2018); safe_delete(c2020)
    n_processed+=1
    print(f'\n  [DONE] {grid.upper()} in {fmt_s(tnow()-loop_t0)} | mem={mem_gb():.2f}GB')

print(f'\nLoop complete — processed:{n_processed} skip_2018:{n_skip_2018} skip_2020:{n_skip_2020}')

## 8. Write CSV Outputs

In [ ]:
write_csv(CSV_DIR/'c2c_summary_stats.csv',      all_summaries)
write_csv(CSV_DIR/'c2c_threshold_fractions.csv', all_thresholds)
write_csv(CSV_DIR/'c2c_statistical_tests.csv',   all_tests)
write_csv(CSV_DIR/'c2c_extreme_values.csv',      all_extreme)
write_csv(CSV_DIR/'normalisation_checks.csv',    all_norm)

cc_rows=[]
for r in all_cc_results:
    if r.get('status')=='ok' and 'epoch_comparison' in r:
        ec=r['epoch_comparison']
        cc_rows.append({'grid_id':r['grid_id'],
            'n_gaps_damage':ec.get('n_gaps_damage'),'n_gaps_recovery':ec.get('n_gaps_recovery'),
            'mean_area_damage':ec.get('mean_gap_area_damage_m2'),'mean_area_recovery':ec.get('mean_gap_area_recovery_m2'),
            'total_area_damage':ec.get('total_damaged_area_damage_m2'),'total_area_recovery':ec.get('total_damaged_area_recovery_m2'),
            'alpha_damage':ec.get('power_law_damage',{}).get('alpha_mle'),
            'alpha_recovery':ec.get('power_law_recovery',{}).get('alpha_mle'),
            'delta_alpha':ec.get('power_law_comparison',{}).get('delta_alpha'),
            'ks_pvalue':ec.get('ks_2samp',{}).get('pvalue'),
            'mw_pvalue':ec.get('mann_whitney_u',{}).get('pvalue'),
            'interpretation':ec.get('power_law_comparison',{}).get('interpretation')})
write_csv(CSV_DIR/'connected_component_epoch_comparison.csv',cc_rows)
print('All CSVs written.')

## 9. Results Review

In [ ]:
import pandas as pd
summary_df=pd.read_csv(CSV_DIR/'c2c_summary_stats.csv')
cc_df=pd.read_csv(CSV_DIR/'connected_component_epoch_comparison.csv')
print('=== C2C Summary (damage 2017→2018) ===')
print(summary_df[summary_df['interval']=='2017-2018'][['grid','n_clean','mean','median','p95']].to_string(index=False))
print('\n=== Power Law Exponents ===')
print(cc_df[['grid_id','alpha_damage','alpha_recovery','delta_alpha','interpretation']].to_string(index=False))
print('\n=== Gap Counts & Areas ===')
print(cc_df[['grid_id','n_gaps_damage','n_gaps_recovery','mean_area_damage','mean_area_recovery']].to_string(index=False))